# Freddie Mac Mortgage Risk Project
## Notebook 02: Target Creation

This notebook constructs the business target for the Freddie Mac mortgage risk model. The objective is to identify whether a loan becomes seriously delinquent (90+ days past due) within 12 months of first payment date, using the combined multi-year origination and servicing datasets created in Notebook 01.

## 1. Import packages and define project paths


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/freddie_mac_risk_project")
PROCESSED = BASE / "data" / "processed"
TABLES = BASE / "outputs" / "tables"

print("Processed path exists:", PROCESSED.exists())
print("Tables path exists:", TABLES.exists())

Mounted at /content/drive
Processed path exists: True
Tables path exists: True


## 2. Load combined typed origination and servicing datasets


In [2]:
orig_all_typed = pd.read_parquet(PROCESSED / "orig_all_typed_2019_2023.parquet")
svcg_all_typed = pd.read_parquet(PROCESSED / "svcg_all_typed_2019_2023.parquet")

print("Origination shape:", orig_all_typed.shape)
print("Servicing shape:", svcg_all_typed.shape)

Origination shape: (250000, 33)
Servicing shape: (9491729, 32)


## 3. Select target-construction fields

This section keeps only the columns required to construct the 12-month serious delinquency target.

In [3]:
orig_target_base = orig_all_typed[[
    "loan_sequence_number",
    "vintage_year",
    "first_payment_date"
]].copy()

svcg_target_base = svcg_all_typed[[
    "loan_sequence_number",
    "vintage_year",
    "monthly_reporting_period",
    "current_loan_delinquency_status"
]].copy()

print("Origination target base shape:", orig_target_base.shape)
print("Servicing target base shape:", svcg_target_base.shape)

Origination target base shape: (250000, 3)
Servicing target base shape: (9491729, 4)


## 4. Merge first payment date into servicing records

In [4]:
svcg_target = svcg_target_base.merge(
    orig_target_base[["loan_sequence_number", "first_payment_date"]],
    on="loan_sequence_number",
    how="left"
)

print("Merged servicing target shape:", svcg_target.shape)
svcg_target.head()

Merged servicing target shape: (9491729, 5)


,loan_sequence_number,vintage_year,monthly_reporting_period,current_loan_delinquency_status,first_payment_date
0,F19Q10000056,2019,2019-02-01,0,2019-03-01
1,F19Q10000056,2019,2019-03-01,0,2019-03-01
2,F19Q10000056,2019,2019-04-01,0,2019-03-01
3,F19Q10000056,2019,2019-05-01,0,2019-03-01
4,F19Q10000056,2019,2019-06-01,0,2019-03-01


## 5. Validate first payment date merge

In [5]:
merge_quality = pd.DataFrame({
    "metric": [
        "servicing_rows",
        "servicing_rows_with_first_payment_date",
        "missing_first_payment_date"
    ],
    "value": [
        len(svcg_target),
        svcg_target["first_payment_date"].notna().sum(),
        svcg_target["first_payment_date"].isna().sum()
    ]
})

merge_quality

,metric,value
0,servicing_rows,9491729
1,servicing_rows_with_first_payment_date,9491729
2,missing_first_payment_date,0


## 6. Calculate months since first payment

In [6]:
svcg_target["months_from_first_payment"] = (
    (svcg_target["monthly_reporting_period"].dt.year - svcg_target["first_payment_date"].dt.year) * 12
    + (svcg_target["monthly_reporting_period"].dt.month - svcg_target["first_payment_date"].dt.month)
)

svcg_target[[
    "loan_sequence_number",
    "first_payment_date",
    "monthly_reporting_period",
    "months_from_first_payment"
]].head(10)

,loan_sequence_number,first_payment_date,monthly_reporting_period,months_from_first_payment
0,F19Q10000056,2019-03-01,2019-02-01,-1
1,F19Q10000056,2019-03-01,2019-03-01,0
2,F19Q10000056,2019-03-01,2019-04-01,1
3,F19Q10000056,2019-03-01,2019-05-01,2
4,F19Q10000056,2019-03-01,2019-06-01,3
5,F19Q10000056,2019-03-01,2019-07-01,4
6,F19Q10000056,2019-03-01,2019-08-01,5
7,F19Q10000056,2019-03-01,2019-09-01,6
8,F19Q10000056,2019-03-01,2019-10-01,7
9,F19Q10000056,2019-03-01,2019-11-01,8


## 7. Restrict servicing history to the first 12 months

In [7]:
svcg_target_12m = svcg_target[
    (svcg_target["months_from_first_payment"] >= 0) &
    (svcg_target["months_from_first_payment"] <= 11)
].copy()

print("12-month servicing subset shape:", svcg_target_12m.shape)

12-month servicing subset shape: (2891849, 6)


## 7A. Ensure delinquency status is numeric

In [9]:
svcg_target_12m["current_loan_delinquency_status"] = pd.to_numeric(
    svcg_target_12m["current_loan_delinquency_status"],
    errors="coerce"
)

print(svcg_target_12m["current_loan_delinquency_status"].dtype)
print(svcg_target_12m["current_loan_delinquency_status"].value_counts(dropna=False).head(10))

float64
current_loan_delinquency_status
0.0    2859376
1.0      18056
2.0       5012
3.0       2947
4.0       2014
5.0       1449
6.0       1018
7.0        728
8.0        522
9.0        335
Name: count, dtype: int64


## 8. Define monthly serious delinquency flag

In [10]:
svcg_target_12m["serious_dq_month_flag"] = (
    svcg_target_12m["current_loan_delinquency_status"] >= 3
).astype(int)

svcg_target_12m[[
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_loan_delinquency_status",
    "serious_dq_month_flag"
]].head(10)

,loan_sequence_number,monthly_reporting_period,current_loan_delinquency_status,serious_dq_month_flag
1,F19Q10000056,2019-03-01,0.0,0
2,F19Q10000056,2019-04-01,0.0,0
3,F19Q10000056,2019-05-01,0.0,0
4,F19Q10000056,2019-06-01,0.0,0
5,F19Q10000056,2019-07-01,0.0,0
6,F19Q10000056,2019-08-01,0.0,0
7,F19Q10000056,2019-09-01,0.0,0
8,F19Q10000056,2019-10-01,0.0,0
9,F19Q10000056,2019-11-01,0.0,0
10,F19Q10000056,2019-12-01,0.0,0


## 9. Aggregate monthly delinquency to a loan-level target

In [11]:
loan_target = (
    svcg_target_12m.groupby("loan_sequence_number", as_index=False)["serious_dq_month_flag"]
    .max()
    .rename(columns={"serious_dq_month_flag": "target_12m_serious_dq"})
)

print("Loan-level target shape:", loan_target.shape)
loan_target.head()

Loan-level target shape: (249799, 2)


,loan_sequence_number,target_12m_serious_dq
0,F19Q10000056,0
1,F19Q10000060,0
2,F19Q10000084,0
3,F19Q10000106,0
4,F19Q10000189,0


## 10. Merge final target to origination dataset

In [12]:
modeling_base = orig_all_typed.merge(
    loan_target,
    on="loan_sequence_number",
    how="left"
)

modeling_base["target_12m_serious_dq"] = modeling_base["target_12m_serious_dq"].fillna(0).astype(int)

print("Modeling base shape:", modeling_base.shape)

display(modeling_base[[
    "loan_sequence_number",
    "vintage_year",
    "first_payment_date",
    "credit_score",
    "original_upb",
    "target_12m_serious_dq"
]].head())

Modeling base shape: (250000, 34)


,loan_sequence_number,vintage_year,first_payment_date,credit_score,original_upb,target_12m_serious_dq
0,F19Q10000056,2019,2019-03-01,741,372000,0
1,F19Q10000060,2019,2019-03-01,731,250000,0
2,F19Q10000084,2019,2019-03-01,722,261000,0
3,F19Q10000106,2019,2019-03-01,729,61000,0
4,F19Q10000189,2019,2019-03-01,773,390000,0


## 11. Review target distribution

In [13]:
target_distribution = modeling_base["target_12m_serious_dq"].value_counts(dropna=False).sort_index()
target_rate = modeling_base["target_12m_serious_dq"].mean()

print("Target distribution:")
print(target_distribution)

print("\nTarget rate:", target_rate)

Target distribution:
target_12m_serious_dq
0    247394
1      2606
Name: count, dtype: int64

Target rate: 0.010424


## 12. Save target-construction outputs


In [14]:
loan_target.to_parquet(PROCESSED / "loan_target_12m_serious_dq.parquet", index=False)
modeling_base.to_parquet(PROCESSED / "modeling_base_with_target_2019_2023.parquet", index=False)

target_summary = pd.DataFrame({
    "metric": ["loan_count", "target_rate"],
    "value": [len(modeling_base), modeling_base["target_12m_serious_dq"].mean()]
})

target_summary.to_csv(TABLES / "target_summary_12m_serious_dq.csv", index=False)

print(PROCESSED / "loan_target_12m_serious_dq.parquet")
print(PROCESSED / "modeling_base_with_target_2019_2023.parquet")
print(TABLES / "target_summary_12m_serious_dq.csv")

/content/drive/MyDrive/freddie_mac_risk_project/data/processed/loan_target_12m_serious_dq.parquet
/content/drive/MyDrive/freddie_mac_risk_project/data/processed/modeling_base_with_target_2019_2023.parquet
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/target_summary_12m_serious_dq.csv


## Conclusion

The 12-month serious delinquency target has been successfully constructed from the Freddie Mac multi-year servicing data and merged back to the origination dataset. The resulting modeling base now contains origination-stage loan features and a clean binary target suitable for exploratory analysis and predictive modeling.